# Notebook EDA, Preprocessing, Feature Engineering & Pemodelan Preeklamsia
**Proyek Terintegrasi Data Analitik & Data Science — Institut Teknologi Del**  
**Topik:** Analisis Faktor Risiko dan Prediksi Risiko Preeklamsia pada Ibu Hamil di Kabupaten Toba  
**Dataset:** `dataset_sintetis_preeklamsia.csv` (1.000 baris kunjungan ANC, 250 ibu hamil unik, 103 variabel master)

In [1]:
import pandas as pd
import numpy as np
import json
import sqlite3

df_raw = pd.read_csv('../data/dataset_sintetis_preeklamsia.csv')
print(f'Ukuran Dataset Mentah: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom')
df_raw.head(3)

## 1. Audit Kualitas Data & Penanganan Nilai Sentinel (`-1` dan `TIDAK_DIPERIKSA`)
Sesuai **Master Variabel Bab 2 & Bab 7**, nilai `-1` pada kolom numerik dan `TIDAK_DIPERIKSA` pada kolom kategorikal bukan merupakan nilai pengukuran, melainkan penanda bahwa pemeriksaan belum/tidak dilakukan. Kita memisahkan `'tidak diperiksa'` dari `'hasil negatif'`.

In [2]:
with open('../data/data_quality_report.json') as f:
    quality_report = json.load(f)
print(json.dumps(quality_report, indent=2))

## 2. Lima Temuan Utama Exploratory Data Analysis (EDA)
1. **Prevalensi Preeklamsia Daerah:** Tercatat **165 kunjungan (16,5%)** atau **42 dari 250 ibu hamil (16,8%)** terdiagnosis akhir `PREECLAMPSIA`.
2. **Faktor Risiko Komorbiditas Dominan:** Sindrom Antifosfolipid (100%), Hipertensi Kronik (66,0%), Penyakit Ginjal (62,5%), dan Riwayat Preeklamsia Sebelumnya (51,3%) meningkatkan risiko secara drastis.
3. **Parameter Hemodinamik:** `mean_arterial_pressure_mmHg` ($r = 0,539$) dan `systolic_bp_mmHg` ($r = 0,526$) berkorelasi positif kuat dengan diagnosis preeklamsia.
4. **Kesenjangan Uji Proteinuria:** 73,1% kunjungan belum diperiksa protein urine (`TIDAK_DIPERIKSA`), padahal 100% pasien dengan hasil dipstick `2+` dan `3+` positif mengalami preeklamsia.
5. **Dampak Luaran Bayi:** Preeklamsia berkorelasi negatif dengan usia kehamilan saat lahir ($r = -0,637$) dan berat lahir bayi ($r = -0,512$).

In [3]:
df_clean = pd.read_csv('../data/cleaned_preeklamsia_toba.csv')
print('Distribusi Target Diagnosis Akhir:')
print(df_clean['final_diagnosis'].value_counts())
print('\nRata-rata MAP & Berat Bayi berdasarkan Diagnosis:')
print(df_clean.groupby('final_diagnosis')[['systolic_bp_mmHg', 'diastolic_bp_mmHg', 'mean_arterial_pressure_mmHg', 'birth_weight_g']].mean().round(2))

## 3. Evaluasi 4 Model Machine Learning (GroupKFold 5-Fold Tanpa Data Leakage)

In [4]:
with open('../models/model_evaluation_metrics.json') as f:
    metrics = json.load(f)

summary_df = pd.DataFrame([
    {
        'Model': k,
        'Accuracy': v['accuracy'],
        'Precision': v['precision'],
        'Recall': v['recall'],
        'F1-Score': v['f1_score'],
        'ROC-AUC': v['roc_auc']
    }
    for k, v in metrics['models'].items()
])
summary_df